# 🌍 duckOSM Network Explorer

This notebook allows you to interactively explore and visualize the routing networks generated by `duckOSM`.

In [1]:
import duckdb
import leafmap.foliumap as leafmap
import geopandas as gpd
from shapely import wkt
import os

## 1. Connect to Database
Select your transportation mode (e.g., `driving`, `walking`, `cycling`).

In [2]:
db_path = '../data/db/sweden.duckdb'
mode = 'driving'

con = duckdb.connect(db_path, read_only=True)
con.execute("INSTALL spatial; LOAD spatial;")

# Check available schemas
con.execute("SELECT schema_name FROM information_schema.schemata WHERE schema_name NOT IN ('information_schema', 'pg_catalog', 'main', 'raw')").fetchall()

IOException: IO Error: Cannot open database "/home/kaveh/projects/duckOSM/data/db/sweden.duckdb" in read-only mode: database does not exist

## 2. Load Network Data
We load the edges into a GeoDataFrame for visualization. Note that we use `ST_AsText(geometry)` because we are now using native DuckDB spatial types.

In [ ]:
query = f"""
    SELECT 
        edge_id, osm_id, name, highway, 
        length_m, maxspeed_kmh, cost_s, oneway, from_cell, to_cell,
        ST_AsText(geometry) as geometry
    FROM {mode}.edges
    LIMIT 5000
"""
df = con.execute(query).df()

# Convert WKT to geometry
df['geometry'] = df['geometry'].apply(wkt.loads)
gdf = gpd.GeoDataFrame(df, crs="EPSG:4326")

gdf.head()

,edge_id,osm_id,name,highway,length_m,maxspeed_kmh,cost_s,oneway,from_cell,to_cell,geometry
0,1815607,1051918536,None,tertiary,66.461624,70.0,3.418026,False,644168022504094506,644168022507536451,"LINESTRING (14.09832 57.82915, 14.09839 57.829..."
1,1815608,322684964,None,residential,354.501343,70.0,18.231497,False,644563225475932594,644563225464542324,"LINESTRING (16.50176 56.69324, 16.50193 56.693..."
2,1815609,386580083,None,service,14.729112,20.0,2.651240,False,644562519954207637,644562519954209486,"LINESTRING (12.72966 56.07693, 12.72971 56.076..."
3,1815610,36334266,Skolgatan,tertiary,9.417622,50.0,0.678069,False,644164587430538416,644164587430538324,"LINESTRING (16.85735 59.93669, 16.85737 59.93678)"
4,1815611,820450145,Trelleborgsvägen,motorway,150.447861,110.0,4.923748,True,644560497758844206,644560497758815104,"LINESTRING (13.00465 55.54241, 13.005 55.54159..."


## 3. Visualize on Map
Click on any edge to see its properties.

In [ ]:
m = leafmap.Map(center=[gdf.geometry.centroid.y.mean(), gdf.geometry.centroid.x.mean()], zoom=13)

# Simple coloring based on highway type
m.add_gdf(
    gdf, 
    layer_name=f"{mode.capitalize()} Network",
    info_mode="on_click"
)
m

## 4. Administrative boundaries

The `admin_boundaries` table (added by `scripts/add_admin_boundaries.py`) holds OSM
administrative areas — country, counties (*län*), municipalities (*kommuner*) and
districts — each with a derived `parent_osm_id` so you can walk the hierarchy. It
needs a DB built with that script (e.g. the whole-Sweden `sweden.duckdb`).

In [ ]:
# Administrative boundaries live in a DB built with scripts/add_admin_boundaries.py
bdb_path = '../data/db/sweden.duckdb'
bcon = duckdb.connect(bdb_path, read_only=True)
bcon.execute("INSTALL spatial; LOAD spatial;")
bcon.execute(
    "SELECT admin_level, count(*) AS n FROM admin_boundaries "
    "GROUP BY admin_level ORDER BY admin_level NULLS LAST"
).df()

,admin_level,n
0,2,1
1,4,21
2,6,2
3,7,291
4,8,89
5,9,532
6,10,530
7,11,4
8,<NA>,2


### Fuzzy name search

Matches the local `name` **or** the English `name_en` (so `sweden` finds `Sverige`),
case- and diacritic-insensitive (`sodermalm` → `Södermalm`) and typo-tolerant via
`jaro_winkler_similarity`. Each match shows `osm_id`, `admin_level` and parent.

In [ ]:
def search_boundary(query, limit=10):
    """Return admin boundaries whose name (or English name_en) is similar to `query`."""
    sql = """
        WITH q AS (SELECT strip_accents(lower($name)) AS qn)
        SELECT b.osm_id,
               b.name,
               b.name_en,
               b.admin_level,
               p.name AS parent,
               round(greatest(
                   jaro_winkler_similarity(strip_accents(lower(b.name)), q.qn),
                   coalesce(jaro_winkler_similarity(strip_accents(lower(b.name_en)), q.qn), 0)
               ), 3) AS similarity
        FROM admin_boundaries b
        CROSS JOIN q
        LEFT JOIN admin_boundaries p ON b.parent_osm_id = p.osm_id
        WHERE b.name IS NOT NULL
        ORDER BY (strip_accents(lower(b.name)) LIKE '%' || q.qn || '%'
                  OR strip_accents(lower(coalesce(b.name_en, ''))) LIKE '%' || q.qn || '%') DESC,
                 similarity DESC
        LIMIT $limit
    """
    return bcon.execute(sql, {"name": query, "limit": limit}).df()


# English or Swedish, diacritics optional: "sweden" -> Sverige, "sodermalm" -> Sodermalm
search_boundary("sweden")

BinderException: Binder Error: Table "b" does not have a column named "name_en"

Candidate bindings: : "name"

LINE 5:                b.name_en,
                       ^

### Ancestry chain (kommun → county → country)

`parent_osm_id` links each boundary to its immediate parent, so a recursive query
gives the full chain.

In [ ]:
def ancestry(osm_id):
    """Return a boundary and all of its ancestors (closest first)."""
    sql = """
        WITH RECURSIVE up AS (
            SELECT osm_id, name, admin_level, parent_osm_id, 0 AS depth
            FROM admin_boundaries WHERE osm_id = $id
            UNION ALL
            SELECT b.osm_id, b.name, b.admin_level, b.parent_osm_id, up.depth + 1
            FROM admin_boundaries b JOIN up ON b.osm_id = up.parent_osm_id
        )
        SELECT depth, admin_level, name, osm_id FROM up ORDER BY depth
    """
    return bcon.execute(sql, {"id": osm_id}).df()


# Take the top hit for 'sodermalm' and show its ancestry
hit = int(search_boundary("sodermalm").iloc[0]["osm_id"])
ancestry(hit)

,depth,admin_level,name,osm_id
0,0,10,Södermalm,5695402
1,1,9,Södermalms stadsdelsområde,5691336
2,2,7,Stockholms kommun,398021
3,3,4,Stockholms län,54391
4,4,2,Sverige,52822


### Children of a boundary

The inverse — everything whose `parent_osm_id` points at a given boundary (e.g. the
kommuner of a county).

In [ ]:
def children(osm_id, limit=50):
    return bcon.execute(
        "SELECT osm_id, name, admin_level FROM admin_boundaries "
        "WHERE parent_osm_id = $id ORDER BY name LIMIT $limit",
        {"id": osm_id, "limit": limit},
    ).df()


# Kommuner of Stockholms län
county = int(search_boundary("stockholms lan").iloc[0]["osm_id"])
children(county)

,osm_id,name,admin_level
0,398626,Botkyrka kommun,7
1,398034,Danderyds kommun,7
2,398920,Ekerö kommun,7
3,398625,Haninge kommun,7
4,398567,Huddinge kommun,7
5,398022,Järfälla kommun,7
6,398035,Lidingö kommun,7
7,398038,Nacka kommun,7
8,397159,Norrtälje kommun,7
9,935473,Nykvarns kommun,7


### Map a boundary

Load a matched boundary's polygon and show it on a map (reuses `leafmap` from above).

In [ ]:
sel = search_boundary("sodermalm").iloc[0]
bdf = bcon.execute(
    "SELECT osm_id, name, admin_level, ST_AsText(geometry) AS geometry "
    "FROM admin_boundaries WHERE osm_id = $id",
    {"id": int(sel["osm_id"])},
).df()
bdf["geometry"] = bdf["geometry"].apply(wkt.loads)
bgdf = gpd.GeoDataFrame(bdf, crs="EPSG:4326")

bm = leafmap.Map(
    center=[bgdf.geometry.centroid.y.mean(), bgdf.geometry.centroid.x.mean()], zoom=12
)
bm.add_gdf(bgdf, layer_name=sel["name"], info_mode="on_click")
bm

### Spatial: which boundaries contain a point?

Geometries are EPSG:4326 (lon/lat). `admin_level DESC` lists the finest area first.

In [ ]:
# A point in central Stockholm (lon, lat)
bcon.execute(
    "SELECT admin_level, name FROM admin_boundaries "
    "WHERE ST_Contains(geometry, ST_Point($lon, $lat)) ORDER BY admin_level DESC",
    {"lon": 18.0686, "lat": 59.3293},
).df()

,admin_level,name
0,10,Norrmalm
1,9,Norra innerstadens stadsdelsområde
2,7,Stockholms kommun
3,4,Stockholms län
4,2,Sverige


In [ ]:
search_boundary("sweden")

,osm_id,name,admin_level,parent,similarity
0,1517104,Heden,10,Centrum,0.822
1,3050088,Segevång,10,Norr,0.750
2,8033086,Skeda,9,Linköpings kommun,0.730
3,5876305,Näsudden,10,Bullås,0.722
4,52820,Umedalen,10,Backenområdet,0.722
5,16441466,Aspudden,10,Hägersten-Älvsjö stadsdelsområde,0.722
6,6448449,Ösbydalen,10,Gustavsberg,0.704
7,2733413,Bosvedjan,9,Sköns distrikt,0.704
8,307884,Smedjebackens kommun,7,Dalarnas län,0.694
9,6851346,Slädaviken,10,Släda,0.689


In [ ]:
bcon.close()